In [0]:
"""
Basic cleaning and normalization of synthetic healthcare bronze tables into silver.

This notebook:
- Reads bronze Delta tables.
- Applies simple data quality checks and transformations.
- Writes silver tables for patients and encounters (extend as needed).
"""

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


CATALOG_NAME = 'healthcare'
SCHEMA_NAME = "readmission"

In [0]:
def get_spark() -> SparkSession:
    return SparkSession.builder.getOrCreate()


def qualify_table(table_name: str) -> str:
    if CATALOG_NAME and SCHEMA_NAME:
        return f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
    if SCHEMA_NAME:
        return f"{SCHEMA_NAME}.{table_name}"
    return table_name

In [0]:
def main():
    spark = get_spark()

    if CATALOG_NAME and SCHEMA_NAME:
        spark.sql(f"USE CATALOG healthcare")
        spark.sql(f"USE SCHEMA readmission")
    elif SCHEMA_NAME:
        spark.sql(f"USE SCHEMA readmission")

    bronze_patients = spark.table(qualify_table("bronze_patients"))
    bronze_encounters = spark.table(qualify_table("bronze_encounters"))

    patients_silver = (
        bronze_patients.withColumn("age", F.when(F.col("age") < 0, None).otherwise(F.col("age")))
        .withColumn("gender", F.upper(F.col("gender")))
        .dropDuplicates(["patient_id"])
    )

    encounters_silver = bronze_encounters.dropDuplicates(["encounter_id"]).withColumn(
        "length_of_stay",
        F.when(F.col("length_of_stay") < 0, None).otherwise(F.col("length_of_stay")),
    )

    patients_silver.write.mode("overwrite").format("delta").saveAsTable(qualify_table("patients_silver"))
    encounters_silver.write.mode("overwrite").format("delta").saveAsTable(qualify_table("encounters_silver"))

    print("Silver tables created/updated:")
    print(f"- {qualify_table('patients_silver')}")
    print(f"- {qualify_table('encounters_silver')}")


if __name__ == "__main__":
    main()

In [0]:
%sql
select * from patients_silver

In [0]:
%sql
select * from encounters_silver